# Tutorial: Evaluating pertTF on the 50-Gene CRISPRi Dataset

This tutorial walks through loading a pre-trained `pertTF` model from HuggingFace,
preparing the 50-gene CRISPRi dataset, running perturbation prediction inference,
and visualizing results with UMAP embeddings and gene-expression heatmaps.

**Target model**: [`weililab/perttf-tiny-perturb-5k-nb`](https://huggingface.co/weililab/perttf-tiny-perturb-5k-nb)  
**Target dataset**: [`weililab/CRISPRI_50TF`](https://huggingface.co/datasets/weililab/CRISPRI_50TF)  
**Companion script**: `evaluate_50gene_hf.py` (the full runnable version of this tutorial)

---

## Prerequisites

install pertTF from github

Key packages:
- `pertTF` — the perturbation transformer package (available at `pertTF/` in the repo)
- `cell_eval` — installed separately for DEG and metric computation
- `huggingface_hub` — for model and dataset downloads
- `anndata`, `scanpy` — single-cell data handling
- `umap-learn`, `seaborn`, `matplotlib`, `adjustText` — visualization

### GPU

A CUDA-capable GPU is required. The model uses FlashAttention and
mixed-precision inference. 



---

## Imports and Setup

All imports needed for the tutorial. Note: no imports from `pertx` — all helper
functions are self-contained within this script.

In [1]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import anndata as ad, scanpy as sc
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns
from scipy.sparse import issparse
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf
import umap

# Add pertTF to path
REPO_ROOT = Path.cwd()  # or Path(__file__).resolve().parents[5]
sys.path.insert(0, str("/autofs/projects-t3/lilab/yangqisu/repos/pertf_benchmark/pertTF"))
sys.path.insert(0, str(REPO_ROOT))

from perttf.model.hf import HFPerturbationTFModel
from perttf.model.train_function import eval_testdata
from perttf.utils.logger import create_logger
from cell_eval import MetricsEvaluator

warnings.filterwarnings("ignore")

## Helper Functions

In [58]:
def filter_pred_by_true_combinations(
    adata_true, adata_pred,
    context_col="celltype", pert_col="genotype",
    ctrl_value="WT", min_cells=30,
):
    """Filters adata_pred to match valid (context, pert) combinations in adata_true."""
    counts = adata_true.obs.groupby([context_col, pert_col]).size().reset_index(name="count")
    valid_perts = counts[
        (counts[pert_col] == ctrl_value) | (counts["count"] >= min_cells)
    ]

    def has_experiment_structure(group):
        has_ctrl = ctrl_value in group[pert_col].values
        has_treated = any(group[pert_col] != ctrl_value)
        return has_ctrl and has_treated

    valid_contexts = valid_perts.groupby(context_col).filter(has_experiment_structure)

    def get_key(df):
        return df[context_col].astype(str) + "_" + df[pert_col].astype(str)

    valid_keys_set = set(get_key(valid_contexts))
    true_mask = get_key(adata_true.obs).isin(valid_keys_set)
    adata_true_filtered = adata_true[true_mask].copy()

    true_combos = set(get_key(adata_true_filtered.obs).unique())
    pred_mask = get_key(adata_pred.obs).isin(true_combos)
    adata_pred_filtered = adata_pred[pred_mask].copy()

    return adata_true_filtered, adata_pred_filtered


def create_perturb_anndata_pair_for_cell_eval(
    adata_true, adata_pred,
    pert_col="genotype", context_col="celltype",
    sample=False, top_k=None, ctrl_value="WT", min_eval_cells=30,
):
    """Build paired (raw, model) AnnData for cell_eval MetricsEvaluator."""
    ctrl_mask = adata_true.obs[pert_col] == ctrl_value
    adata_true_ctrl_expr = adata_true[ctrl_mask].X.toarray() if issparse(adata_true.X) else adata_true[ctrl_mask].X
    ctrl_expr_context = adata_true[ctrl_mask].obs[context_col].values

    pert_inds = np.where(adata_true.obs[pert_col].isin(adata_pred.obs.genotype_next.unique()))[0]
    ctrl_inds = np.where(adata_true.obs[pert_col] == ctrl_value)[0]
    pert_true_adata = adata_true[np.concatenate([pert_inds, ctrl_inds])].copy()

    cell_names_A = [f"Cell_{i}" for i in range(adata_true_ctrl_expr.shape[0])]
    obs_A = pd.DataFrame({context_col: ctrl_expr_context, pert_col: ctrl_value}, index=cell_names_A)
    adata_A = ad.AnnData(X=adata_true_ctrl_expr, obs=obs_A)
    adata_A.var_names = adata_true.var.index

    if sample:
        mask = (
            np.random.rand(*adata_pred.obsm["mvc_next_expr_zero"].shape)
            < adata_pred.obsm["mvc_next_expr_zero"]
        ).astype(int)
    else:
        mask = 1.0
    pert_pred_expr = adata_pred.obsm["mvc_next_expr"] * mask
    pert_pred_expr[pert_pred_expr < 0] = 0

    perturbation_label = adata_pred.obs.genotype_next.values
    pred_ctrl_expr_context = adata_pred.obs[context_col].values
    cell_names_B = [f"Cell_{i}" for i in range(pert_pred_expr.shape[0])]
    obs_B = pd.DataFrame({context_col: pred_ctrl_expr_context, pert_col: perturbation_label}, index=cell_names_B)

    adata_B = ad.AnnData(X=pert_pred_expr, obs=obs_B)
    adata_B.var_names = adata_true.var.index

    pert_pred_adata = ad.concat({"raw": adata_A, "model": adata_B}, label="data_source", index_unique="-")
    pert_true_adata, pert_pred_adata = filter_pred_by_true_combinations(
        pert_true_adata, pert_pred_adata,
        context_col=context_col, pert_col=pert_col,
        ctrl_value=ctrl_value, min_cells=min_eval_cells,
    )

    if top_k is not None:
        assert isinstance(top_k, int), "top_k must be int or None"
        if isinstance(adata_true.X, np.ndarray):
            means = adata_true.X.mean(axis=0)
        else:
            means = np.array(adata_true.X.mean(axis=0)).flatten()
        mean_series = pd.Series(means, index=adata_true.var_names)
        top_genes = mean_series.sort_values(ascending=False).head(top_k)
        return pert_true_adata[:, top_genes.index].copy(), pert_pred_adata[:, top_genes.index].copy()

    return pert_true_adata, pert_pred_adata


def sample_context_ctrl(
    contexts, adata,
    condition_col="condition", context_col="celltype",
    ctrl="WT", total_size=None, min_context_size=300,
):
    """Randomly sample control cells for each context."""
    context_list = []
    if total_size is not None:
        min_context_size = total_size // len(contexts)
    for c in contexts:
        c_inds = np.where(
            (adata.obs[context_col] == c) & (adata.obs[condition_col] == ctrl)
        )[0]
        if len(c_inds) == 0:
            continue
        inds = np.random.choice(c_inds, min(min_context_size, len(c_inds)), replace=True)
        context_list.append(inds)
    if not context_list:
        return np.array([], dtype=int)
    return np.unique(np.concatenate(context_list))


def create_external_testdata_perttf(
    adata, context_col="celltype", ctrl="WT", pert_col="genotype", min_c_size=300,
):
    """Build a test AnnData with genotype_next labels for perturbation prediction."""
    test_set = [g for g in adata.obs[pert_col].unique() if g != ctrl]
    test_df = adata[adata.obs[pert_col].isin(test_set)].obs
    test_contexts = test_df[context_col].unique()
    print(f"  External test contexts: {len(test_contexts)}")

    test_ind_list = [np.where(adata.obs[pert_col].isin(test_set))[0]]
    genotype_next_list = adata[test_ind_list[0]].obs[pert_col].tolist()

    for p in test_set:
        print(f"    sampling non-perturbed for {p}")
        test_ctrl_inds = sample_context_ctrl(
            test_contexts, adata,
            context_col=context_col, condition_col=pert_col,
            ctrl=ctrl, min_context_size=min_c_size,
        )
        print(f"      sampled {len(test_ctrl_inds)} control cells")
        test_ind_list.append(test_ctrl_inds)
        genotype_next_list.extend([p] * len(test_ctrl_inds))

    test_indices = np.concatenate(test_ind_list)
    adata_test = adata[test_indices].copy()
    adata_test.obs["genotype_next"] = genotype_next_list
    return adata_test


def run_cell_eval(
    adata_real, adata_pert, outdir,
    context_col="celltype", pert_col="genotype",
    control_pert="WT", sample=False, top_k=None,
):
    """Run cell_eval MetricsEvaluator and return (real, pred) AnnData pair."""
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    real, pred = create_perturb_anndata_pair_for_cell_eval(
        adata_real, adata_pert,
        sample=sample, context_col=context_col,
        pert_col=pert_col, top_k=top_k, ctrl_value=control_pert
    )

    results = {}
    for c in real.obs[context_col].unique():
        pred_sub = pred[pred.obs[context_col] == c].copy()
        real_sub = real[real.obs[context_col] == c].copy()
        prefix = c if isinstance(c, str) else f"celltype_{c}"
        evaluator = MetricsEvaluator(
            adata_pred=pred_sub,
            adata_real=real_sub,
            control_pert=control_pert,
            pert_col=pert_col,
            outdir=outdir,
            num_threads=16,
            prefix=prefix,
        )
        results[prefix] = evaluator.compute()
        print(f"    {prefix}: {results[prefix][1]}")

    return dict(result=results, real=real, pred=pred)


def evaluate_external_data(
    model, true_adata, test_adata, data_produced, config,
    outdir, model_name, data_name,
    sample=False, context_col="celltype", pert_col="genotype",
    ctrl="WT", top_k=None, nb_sf=True,
):
    """Run eval_testdata + cell_eval on external test data."""
    res_dir = Path(outdir) / f"{model_name}_external_{data_name}"
    if top_k is not None:
        res_dir = Path(str(res_dir) + f"_Top{top_k}")
    res_dir.mkdir(parents=True, exist_ok=True)

    print(f"  Running eval_testdata...")
    test_adata = eval_testdata(
        model,
        adata_t=test_adata,
        gene_ids=data_produced["gene_ids"],
        train_data_dict=data_produced,
        config=config,
        include_types=["cls"],
        logger=create_logger(),
        predict_expr=True,
        mvc_full_expr=True,
        sizefactor=nb_sf,
        sample=True,
    )

    # NB size-factor log-transform
    if model.distribution is not None and model.distribution != "zig":
        sf = np.sum(test_adata.obsm["mvc_next_expr"], axis=1) / np.median(
            np.sum(test_adata.obsm["mvc_next_expr"], axis=1)
        )
        test_adata.obsm["mvc_next_expr"] = np.log(
            test_adata.obsm["mvc_next_expr"] / sf.reshape(-1, 1) + 1
        )

    print(f"  Running cell_eval...")
    results = run_cell_eval(
        true_adata, test_adata, res_dir,
        context_col=context_col, pert_col=pert_col,
        control_pert=ctrl, sample=sample, top_k=top_k,
    )

    return test_adata, results


def get_s_t_p(adata_test, ctrl=None, ctrl_label="WT"):
    if ctrl is None:
        adata_test_pert = adata_test[adata_test.obs.genotype == ctrl_label].copy()
        adata_test_s = adata_test[adata_test.obs.genotype == ctrl_label].copy()
    else:
        adata_test_pert = ctrl.copy()
        adata_test_s = ctrl.copy()

    adata_test_t = adata_test[adata_test.obs.genotype != ctrl_label].copy()
    adata_test_s.obs["genotype_next"] = [ctrl_label] * adata_test_s.shape[0]
    adata_test_t.obs["genotype_next"] = adata_test_t.obs.genotype

    t_unique = adata_test_t.obs.genotype.unique()
    n_pert = min(adata_test_pert.shape[0], 100 * len(t_unique))
    pert_next = np.concatenate([
        np.random.choice(t_unique, n_pert),
        np.random.choice(t_unique, max(0, adata_test_pert.shape[0] - n_pert)),
    ])
    adata_test_pert.obs["genotype_next"] = np.random.permutation(pert_next)

    return adata_test_s, adata_test_t, adata_test_pert


def embed_external(model, test, data_produced, config, outdir, model_name, ctrl_label="WT"):
    """Run source/target/pert inference and produce UMAP plots.
    Falls back to direct-embedding UMAP if s/t/p split produces empty subsets
    (e.g., when ctrl_label not in model's genotype_to_index)."""
    out = {}
    emb_dir = Path(outdir)
    emb_dir.mkdir(parents=True, exist_ok=True)

    s, t, p = get_s_t_p(test, ctrl=None, ctrl_label=ctrl_label)

    # Check if control label is in genotype_to_index (determines if s/t/p will work)
    ctrl_in_geno = ctrl_label in data_produced["genotype_to_index"]
    print(f"  Control '{ctrl_label}' in genotype_to_index: {ctrl_in_geno}")

    if ctrl_in_geno and s.shape[0] > 0:
        # Full s/t/p embed
        print("  Running embed inference (pert)...")
        eval_results_p = eval_testdata(
            model, adata_t=p, gene_ids=data_produced["gene_ids"],
            train_data_dict=data_produced, config=config,
            include_types=["cls"], logger=create_logger(),
            predict_expr=True, mvc_full_expr=True,
        )
        print("  Running embed inference (source)...")
        eval_results_s = eval_testdata(
            model, adata_t=s, gene_ids=data_produced["gene_ids"],
            train_data_dict=data_produced, config=config,
            include_types=["cls"], logger=create_logger(),
            predict_expr=True, mvc_full_expr=True,
        )
        print("  Running embed inference (target)...")
        eval_results_t = eval_testdata(
            model, adata_t=t, gene_ids=data_produced["gene_ids"],
            train_data_dict=data_produced, config=config,
            include_types=["cls"], logger=create_logger(),
            predict_expr=True, mvc_full_expr=True,
        )
        for label, adata_obj in [("s", eval_results_s), ("t", eval_results_t), ("p", eval_results_p)]:
            if adata_obj.shape[0] > 0:
                np.save(emb_dir / f"X_scGPT_next_{label}.npy", adata_obj.obsm["X_scGPT_next"])

        print("  Generating UMAP plots...")
        plot_insilico_perturbations2(
            eval_results_s, eval_results_t, eval_results_p,
            save_dir=str(emb_dir / f"{model_name}_50gene"),
            name="PERTTF 50gene",
        )

        out["50gene"] = {"pert_emb": {}, "s_emb": {}, "t_emb": {}}
        for gene in eval_results_p.obs.genotype_next.unique():
            mask_p = eval_results_p.obs.genotype_next == gene
            out["50gene"]["pert_emb"][gene] = eval_results_p.obsm["X_scGPT_next"][mask_p]
            out["50gene"]["s_emb"][gene] = eval_results_s.obsm["X_scGPT_next"][mask_p]
            mask_t = eval_results_t.obs.genotype_next == gene
            if mask_t.any():
                out["50gene"]["t_emb"][gene] = eval_results_t.obsm["X_scGPT_next"][mask_t]
    else:
        # Fallback: use embeddings from the test data directly, colored by genotype_next
        print("  Fallback: using test-data embeddings directly for UMAP (ctrl_label not in genotype_to_index)")
        import umap
        emb_key = "X_scGPT_next"
        embeddings = test.obsm[emb_key] if emb_key in test.obsm else np.zeros((test.shape[0], 2))
        obs = test.obs

        np.save(emb_dir / f"X_scGPT_next.npy", embeddings)
        pd.DataFrame(obs).to_csv(emb_dir / "obs.csv")

        # Simple UMAP colored by genotype_next
        reducer = umap.UMAP(n_neighbors=50, min_dist=0.05, random_state=42)
        emb_2d = reducer.fit_transform(embeddings)
        plot_df = pd.DataFrame(emb_2d, columns=["UMAP1", "UMAP2"])
        plot_df["genotype_next"] = obs.genotype_next.values

        plt.figure(figsize=(14, 10))
        top_pert = list(plot_df["genotype_next"].value_counts().head(10).index)
        sub = plot_df[plot_df["genotype_next"].isin(top_pert)].sample(frac=1, random_state=42)
        sns.scatterplot(data=sub, x="UMAP1", y="UMAP2", hue="genotype_next",
                       palette="tab20", alpha=0.5, linewidth=0)
        plt.title("PERTTF 50gene Embeddings by Predicted Perturbation")
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
        sns.despine()
        plt.grid(False)
        plt.tight_layout()
        umap_path = emb_dir / f"{model_name}_50gene" / "umap_global.png"
        umap_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(umap_path, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"  UMAP saved: {umap_path}")

    return out


# ══════════════════════════════════════════════════════════════════════════════
# 2. UMAP plotting
# ══════════════════════════════════════════════════════════════════════════════

def plot_insilico_perturbations2(
    ctrl_no_pert, pert_true, pert_insilico,
    save_dir=None, n_neighbors=50, min_dist=0.05,
    name="", wt_downsample_frac=0.2, label_overlay=False,
):
    """Generate per-gene + global + split UMAP plots."""
    import umap
    from adjustText import adjust_text

    data_map = {}
    emb_key = "X_scGPT_next"
    obs_col = "genotype_next"

    unique_perts = pert_insilico.obs[obs_col].unique()
    for pert_key in unique_perts:
        s_mask = pert_insilico.obs[obs_col] == pert_key
        mask = pert_true.obs[obs_col] == pert_key
        if mask.sum() == 0:
            continue
        data_map[pert_key] = (
            ctrl_no_pert[s_mask].obsm[emb_key],
            pert_true[mask].obsm[emb_key],
            pert_insilico[s_mask].obsm[emb_key],
        )

    if not data_map:
        print("  Warning: no data_map entries for UMAP")
        return

    full_mat = []
    full_lab = []
    hue_order_full = ["WT -> WT"]

    for pert_name, (m_ctrl, m_true, m_insilico) in data_map.items():
        combined_matrix = np.vstack([m_ctrl, m_true, m_insilico])
        n_c, n_t, n_i = m_ctrl.shape[0], m_true.shape[0], m_insilico.shape[0]
        labels = (
            ["WT -> WT"] * n_c
            + [f"{pert_name} -> {pert_name}"] * n_t
            + [f"WT -> {pert_name}"] * n_i
        )
        full_mat.append(combined_matrix)
        full_lab.extend(labels)

        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=42)
        embedding = reducer.fit_transform(combined_matrix)
        plot_df = pd.DataFrame(embedding, columns=["UMAP1", "UMAP2"])
        plot_df["Condition"] = labels
        hue_order = ["WT -> WT", f"{pert_name} -> {pert_name}", f"WT -> {pert_name}"]
        hue_order_full.append(f"{pert_name} -> {pert_name}")
        hue_order_full.append(f"WT -> {pert_name}")

        plt.figure(figsize=(10, 7))
        sns.scatterplot(
            data=plot_df.sample(frac=1), x="UMAP1", y="UMAP2",
            hue="Condition", hue_order=hue_order, palette="tab20",
            alpha=0.6, linewidth=0,
        )
        sns.despine()
        plt.grid(False)
        plt.xlabel("UMAP1", fontsize=16)
        plt.ylabel("UMAP2", fontsize=16)
        plt.title(f"{name} {pert_name}", fontsize=20)
        plt.legend(loc="best", fontsize=14, framealpha=0.9)
        plt.tight_layout()

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            safe_name = "".join(c for c in pert_name if c.isalnum() or c in (" ", "-", "_")).strip()
            plt.savefig(os.path.join(save_dir, f"umap_{safe_name}.png"), dpi=300)
            plt.close()
        else:
            plt.show()

    # Global UMAP
    print("  Generating global UMAP...")
    global_mat = np.vstack(full_mat)
    global_df = pd.DataFrame(global_mat)
    global_df["Condition"] = full_lab
    wt_mask = global_df["Condition"] == "WT -> WT"
    others_mask = ~wt_mask
    df_wt = global_df[wt_mask].sample(frac=wt_downsample_frac, random_state=42) if wt_mask.any() else global_df[wt_mask]
    df_others = global_df[others_mask]
    combined_df = pd.concat([df_wt, df_others])

    reducer_global = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, random_state=42)
    global_embeddings = reducer_global.fit_transform(combined_df.drop(columns=["Condition"]).values)
    combined_df["UMAP1"] = global_embeddings[:, 0]
    combined_df["UMAP2"] = global_embeddings[:, 1]

    plt.figure(figsize=(10, 10))
    sns.scatterplot(
        data=combined_df.sample(frac=1), x="UMAP1", y="UMAP2",
        hue="Condition", palette="tab20", alpha=0.4, linewidth=0,
        hue_order=hue_order_full, legend=(not label_overlay),
    )
    plt.title(f"{name}", fontsize=18)
    if not label_overlay:
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper right", fontsize=10)
    sns.despine()
    plt.grid(False)
    plt.xlabel("UMAP1", fontsize=16)
    plt.ylabel("UMAP2", fontsize=16)
    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, "umap_global_combined.png"), dpi=300, bbox_inches="tight")
        plt.close()

    # Split UMAP: predictions vs ground truth
    print("  Generating split UMAPs...")
    unique_genes = sorted(set(
        cond.split(" -> ")[1] for cond in combined_df["Condition"].unique() if cond != "WT -> WT"
    ))
    gene_colors = sns.color_palette("husl", len(unique_genes))
    simple_color_map = {"WT": "lightgrey"}
    for i, gene in enumerate(unique_genes):
        simple_color_map[gene] = gene_colors[i]

    def simplify_label(label):
        if label == "WT -> WT":
            return "WT"
        return label.split(" -> ")[1]
    combined_df["Legend_Name"] = combined_df["Condition"].apply(simplify_label)

    df_insilico = combined_df[
        combined_df["Condition"].str.contains("WT -> ") & (combined_df["Condition"] != "WT -> WT")
    ]
    df_ground_truth = combined_df[
        (combined_df["Condition"] == "WT -> WT") | (~combined_df["Condition"].str.contains("WT -> "))
    ]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10), sharex=True, sharey=True)
    sns.scatterplot(
        data=df_insilico.sample(frac=1, random_state=42), x="UMAP1", y="UMAP2",
        hue="Legend_Name", palette=simple_color_map, alpha=0.8, linewidth=0,
        ax=ax1, legend=False,
    )
    ax1.set_title("In-Silico Predictions (WT -> Gene)", fontsize=20)
    sns.scatterplot(
        data=df_ground_truth.sample(frac=1, random_state=42), x="UMAP1", y="UMAP2",
        hue="Legend_Name", palette=simple_color_map, alpha=0.8, linewidth=0,
        ax=ax2, legend=(not label_overlay),
    )
    ax2.set_title("Ground Truth (WT & Gene -> Gene)", fontsize=20)
    for ax in [ax1, ax2]:
        ax.set_xlabel("UMAP1", fontsize=16)
        ax.set_ylabel("UMAP2", fontsize=16)
        ax.grid(False)
        sns.despine(ax=ax)
    plt.suptitle(f"{name} In Silico vs Ground Truth", fontsize=24, y=1.02)
    plt.tight_layout()
    if not label_overlay:
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0.0, fontsize=14)
    if save_dir:
        plt.savefig(os.path.join(save_dir, "umap_global_split.png"), dpi=300, bbox_inches="tight")
        plt.close()

    print("  UMAP plots saved.")


# ══════════════════════════════════════════════════════════════════════════════
# 3. DEG Heatmap helpers
# ══════════════════════════════════════════════════════════════════════════════

def calculate_deltas(adata, genes, perturbation, reference="WT"):
    """Calculate (cell_expression - mean_reference_expression) per gene."""
    cells_p = adata[adata.obs["genotype"] == perturbation, genes]
    cells_ref = adata[adata.obs["genotype"] == reference, genes]
    expr_p = cells_p.X.toarray() if issparse(cells_p.X) else cells_p.X
    expr_ref = cells_ref.X.toarray() if issparse(cells_ref.X) else cells_ref.X
    ref_mean = expr_ref.mean(axis=0)
    deltas = expr_p - ref_mean
    df = pd.DataFrame(deltas, columns=genes)
    df = df.melt(var_name="Gene", value_name="Expression Change")
    return df


def plot_deg_heatmaps(cell_eval_results, outdir, fdr_threshold=0.01, top_n_genes=20, ctrl_label="WT"):
    """Generate side-by-side boxplots of predicted vs true expression deltas for top DEGs."""
    outdir = Path(outdir)
    # cell_eval writes ESC_real_de.csv in subdirectories (e.g., celltype_X/ESC_real_de.csv)
    deg_csv = outdir / "ESC_real_de.csv"
    if not deg_csv.exists():
        found = list(outdir.rglob("ESC_real_de.csv"))
        if found:
            deg_csv = found[0]
            print(f"  Found ESC_real_de.csv at: {deg_csv}")
        else:
            print(f"  DEG file not found in {outdir}, skipping heatmaps.")
            return

    deg_df = pd.read_csv(deg_csv)
    adata_pred = cell_eval_results["pred"]
    adata_true = cell_eval_results["real"]

    if "fold_change" not in deg_df.columns or "target_mean" in deg_df.columns:
        deg_df["fold_change"] = deg_df["target_mean"] - deg_df["reference_mean"]

    reference_label = ctrl_label
    perturbations = [p for p in adata_true.obs["genotype"].unique() if p != reference_label]
    plots_dir = outdir / "deg_heatmaps"
    plots_dir.mkdir(parents=True, exist_ok=True)

    for p in perturbations:
        print(f"  DEG heatmap for: {p}")
        pert_deg = deg_df[deg_df["target"] == p].copy()
        if pert_deg.empty:
            print(f"    No DEGs found for {p}, skipping.")
            continue

        significant_genes = pert_deg[pert_deg["fdr"] < fdr_threshold].copy()
        if significant_genes.empty:
            print(f"    No genes with FDR < {fdr_threshold} for {p}, skipping.")
            continue

        significant_genes["abs_fc"] = significant_genes["fold_change"].abs()
        top_genes = significant_genes.sort_values("abs_fc", ascending=False).head(top_n_genes)["feature"].tolist()

        df_pred = calculate_deltas(adata_pred, top_genes, p, reference_label)
        df_pred["Source"] = "Predicted"
        df_true = calculate_deltas(adata_true, top_genes, p, reference_label)
        df_true["Source"] = "True"
        plot_data = pd.concat([df_pred, df_true])

        plt.figure(figsize=(15, 8))
        sns.boxplot(
            data=plot_data, x="Gene", y="Expression Change", hue="Source",
            showmeans=True, showfliers=False,
            meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": "5"},
        )
        plt.title(f"Expression Change (Delta) — Top {top_n_genes} Genes — Perturbation: {p}", fontsize=20)
        plt.xticks(rotation=45, ha="right")
        plt.axhline(0, color="red", linestyle="--", alpha=0.5)
        plt.ylabel("Expression Change (Perturbed Cell - Control Average)")
        plt.grid(axis="y", linestyle=":", alpha=0.7)
        plt.tight_layout()
        safe_name = "".join(c for c in p if c.isalnum() or c in (" ", "-", "_")).strip()
        plt.savefig(plots_dir / f"expression_change_{safe_name}.png", dpi=300)
        plt.close()
        print(f"    Saved: {plots_dir / f'expression_change_{safe_name}.png'}")

    print(f"  DEG heatmaps saved to {plots_dir}")



---

## Step 1 — Load the Pre-trained Model from HuggingFace

`HFPerturbationTFModel.from_pretrained()` handles the entire load path: it fetches
`config.json`, `model.safetensors`, `vocab.json`, `training_config.json`, and
`running_parameters.pt` from the Hub, reconstructs the model, and loads weights.

The model carries its training vocabulary, genotype dictionary, cell-type dictionary,
and training config — no local files needed.

In [3]:
HF_MODEL = "weililab/perttf-tiny-perturb-5k-nb"
model = HFPerturbationTFModel.from_pretrained(HF_MODEL)

print(f"vocab: {len(model.vocab)} tokens")                      # ~5057
print(f"genotypes: {len(model.genotype_to_index)} entries")     # ~9854
print(f"cell types: {len(model.cell_type_to_index)} entries")   # ~19
print(f"distribution: {model.training_config.distribution}")    # "nb"

model.cuda()
model.eval()

---

## Step 2 — Prepare the 50-Gene CRISPRi Dataset

### Download

The dataset is hosted on HuggingFace at `weililab/CRISPRI_50TF`. It contains
~107,000 cells with 31,953 genes, pre-filtered for mixscape quality.

In [4]:
HF_DATASET = "weililab/CRISPRI_50TF"
HF_DATASET_FILE = "tf50_mixscape_new.h5ad"

data_path = hf_hub_download(
    repo_id=HF_DATASET, filename=HF_DATASET_FILE, repo_type="dataset",
)
adata_full = sc.read_h5ad(data_path)
print(f"Full shape: {adata_full.shape}")  # (107673, 31953)

### Filtering

Three filters are applied:

1. **ECDF quality filter** — keeps only cells that pass the ECDF threshold (if the
   `ecdf` column exists in `.obs`).
2. **Mixscape filter** — removes cells classified as knockouts that failed mixscape
   (if `mixscape_class` column exists).
3. **Gene-vocabulary intersection** — keeps only the ~4,485 genes present in the
   model's training vocabulary. This is the critical step: the model can only predict
   expression for genes it was trained on.

### Select Top-K Perturbations

Choose the 20 perturbations by mixscape and ECDF. The control label is auto-detected:
`WT` if present, otherwise `non-target` (the CRISPRi dataset's control).

In [60]:
# ECDF filter
ko_gene_order = [
    "SMARCC1", "TCF7L2", "HMGA2", "AFF4", "TCF7L1", "HIF1A", "SMARCA4", "TCF12", 
    "NFIB", "CTNNB1", "SMAD3", "ARID1B", "EZH2", "SMAD4", "REST", "SMARCB1", 
    "HDAC1", "CLOCK", "SALL4", "SMARCD1", "ARNT", "JARID2", "SUZ12", "SMARCD2", 
    "CREBBP", "ARID1A", "SOX2", "EP300", "SMARCC2", "KLF6", "POU5F1", "SMARCA2", 
    "PAX5", "EED", "TCF7", "KLF4", "SMARCD3", "MYC", "TFCP2L1", "RUNX1", 
    "ESRRB", "LEF1", "NANOG", "ARNTL", "BATF", "CD151-1", "CD151-2", "CD55-1", 
    "CD81-1", "CD81-2", "NGFRAP1", "NGFRAP1-1", "OR10J3", "OR2A25", "OR2AG2", 
    "OR2D3", "OR2F1", "OR4D6", "OR4X1", "OR6A2", "OR6T1", "TBX3", "TFRC-1"
]
model_genes = set(model.vocab.stoi.keys())
kogenes = [k for k in ko_gene_order if k in model_genes]
ecdf_filter = True
top_k = 20
genotypes = adata_full.obs["genotype"].unique()
CTRL_LABEL = "WT" if "WT" in genotypes else "non-target" if "non-target" in genotypes else genotypes[0]

if ecdf_filter:
    kogenes = kogenes[:top_k] + [CTRL_LABEL]
    adata = adata_full[adata_full.obs.genotype.isin(kogenes), :]


# Mixscape filter
if "mixscape_class" in adata_full.obs.columns:
    adata = adata[adata.obs["mixscape_class"] != "NP"]

# Gene intersection with model vocab

shared_genes = [g for g in adata.var_names if g in model_genes]

adata = adata[:, shared_genes].copy()
adata.obs['genotype'] = adata.obs.genotype.replace({CTRL_LABEL:'WT'})
print(f"After top perturbation and gene filter: {adata.shape}")  # e.g. (107673, 4485)

### Prepare Layers and HVGs

The `eval_testdata` function needs `X_binned` layer, and a
`highly_variable` column in `.var` for HVG-mode tokenization.

In [62]:
if "X_binned" not in adata.layers:
    adata.layers["X_binned"] = adata.X.copy()


if "highly_variable" not in adata.var.columns:
    sc.pp.highly_variable_genes(adata, n_top_genes=5000)
    adata.var["highly_variable"] = adata.var["highly_variable"].fillna(False)

---

## Step 3 — Build In-Silico Perturbation Inputs

The model's `next_cell_pred_type` is `"pert"`, meaning it predicts what a cell's
expression *would be* after a specified perturbation. To do this, we need a
`genotype_next` column in `.obs` — this tells the model *which* perturbation to
simulate for each cell.

### How `genotype_next` works

For each test perturbation `p`:
- **Perturbed cells** (actual `genotype == p`): `genotype_next = p` (we want the model to
  reconstruct their own perturbation)
- **Control cells** (actual `genotype == CTRL_LABEL`): `genotype_next = p` (we ask the
  model to simulate what would happen if we applied perturbation `p` to a control cell)

The `create_external_testdata_perttf` helper builds this structure automatically:

In [63]:
def create_external_testdata_perttf(adata, context_col="celltype", ctrl="WT",
                                     pert_col="genotype", min_c_size=300):
    test_set = [g for g in adata.obs[pert_col].unique() if g != ctrl]
    test_contexts = adata[adata.obs[pert_col].isin(test_set)].obs[context_col].unique()

    # Start with all perturbed cells
    test_ind_list = [np.where(adata.obs[pert_col].isin(test_set))[0]]
    genotype_next_list = adata[test_ind_list[0]].obs[pert_col].tolist()

    # Add control cells assigned to each test perturbation
    for p in test_set:
        test_ctrl_inds = sample_context_ctrl(test_contexts, adata,
                                              context_col=context_col,
                                              condition_col=pert_col,
                                              ctrl=ctrl,
                                              min_context_size=min_c_size)
        test_ind_list.append(test_ctrl_inds)
        genotype_next_list.extend([p] * len(test_ctrl_inds))

    test_indices = np.concatenate(test_ind_list)
    adata_test = adata[test_indices].copy()
    adata_test.obs["genotype_next"] = genotype_next_list
    return adata_test

adata_test = create_external_testdata_perttf(
    adata, context_col="celltype", ctrl=CTRL_LABEL,
    pert_col="genotype", min_c_size=300,
)
print(f"Test data: {adata_test.shape}")  


### Build `data_produced` and Config

The `eval_testdata` function expects a `train_data_dict` with three keys, all available
from the loaded model:

In [64]:
gene_ids = model.vocab(adata.var_names.tolist())
data_produced = {
    "genotype_to_index": model.genotype_to_index,
    "cell_type_to_index": model.cell_type_to_index,
    "vocab": model.vocab,
    "gene_ids": gene_ids,
}

config = OmegaConf.create(dict(model.training_config))
config.amp = True
config.batch_size = 128
config.max_seq_len = 4001
config.hvg_col = "highly_variable"

---

## Step 4 — Run Inference with `eval_testdata`

The `eval_testdata` function runs the model in inference mode and returns an AnnData
with prediction results stored in `.obsm`:

| Key | Description | Shape |
|-----|-------------|-------|
| `X_scGPT` | Source cell embeddings | (N, 32) |
| `X_scGPT_next` | Next-cell (perturbed) embeddings | (N, 32) |
| `mvc_next_expr` | Predicted full-gene expression | (N, 4485) |
| `mvc_next_expr_zero` | Zero-inflation probabilities | (N, 4485) |

In [65]:
test_adata = eval_testdata(
    model,
    adata_t=adata_test,
    gene_ids=data_produced["gene_ids"],
    train_data_dict=data_produced,
    config=config,
    include_types=["cls"],
    logger=create_logger(),
    predict_expr=True,
    mvc_full_expr=True,      # produce full-gene expression
    sizefactor=True,          # use NB size factors
    sample=True,
)

print(f"X_scGPT_next shape: {test_adata.obsm['X_scGPT_next'].shape}")
print(f"mvc_next_expr shape: {test_adata.obsm['mvc_next_expr'].shape}")

### NB Size-Factor Log Transform

Because this model uses a Negative Binomial distribution (`distribution="nb"`), the
raw `mvc_next_expr` values need a size-factor log transform to be comparable to
log-normalized expression:

In [66]:
if model.distribution is not None and model.distribution != "zig":
    sf = np.sum(test_adata.obsm["mvc_next_expr"], axis=1) / np.median(
        np.sum(test_adata.obsm["mvc_next_expr"], axis=1)
    )
    test_adata.obsm["mvc_next_expr"] = np.log(
        test_adata.obsm["mvc_next_expr"] / sf.reshape(-1, 1) + 1
    )

---

## Step 5 — Evaluate Predictions with `cell_eval`

The `cell_eval` package provides `MetricsEvaluator`, which computes per-perturbation
metrics including differential expression, cosine similarity, and clustering agreement.

The pipeline:
1. `create_perturb_anndata_pair_for_cell_eval` — creates paired (real, predicted) AnnData
   objects with matched cell-type/perturbation combinations
2. `filter_pred_by_true_combinations` — ensures only valid (context, pert) pairs are
   compared
3. `MetricsEvaluator.compute()` — runs per-cell-type evaluation and writes
   `ESC_real_de.csv`

In [68]:
def run_cell_eval(adata_real, adata_pert, outdir, context_col="celltype",
                   pert_col="genotype", control_pert="WT", sample=False):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)

    real, pred = create_perturb_anndata_pair_for_cell_eval(
        adata_real, adata_pert, sample=sample,
        context_col=context_col, pert_col=pert_col,
    )

    results = {}
    print(real.obs[context_col].unique())
    for c in real.obs[context_col].unique():
        pred_sub = pred[pred.obs[context_col] == c].copy()
        real_sub = real[real.obs[context_col] == c].copy()
        prefix = c if isinstance(c, str) else f"celltype_{c}"
        evaluator = MetricsEvaluator(
            adata_pred=pred_sub, adata_real=real_sub,
            control_pert=control_pert, pert_col=pert_col,
            outdir=outdir, num_threads=16, prefix=prefix,
        )
        results[prefix] = evaluator.compute()
        print(f"    {prefix}: {results[prefix][1]}")

    return dict(result=results, real=real, pred=pred)

eval_results = run_cell_eval(
    adata, test_adata,
    Path("results/CELL_EVAL/50gene_hf"),
    context_col="celltype", pert_col="genotype", control_pert='WT',
)

**Note**: The `create_perturb_anndata_pair_for_cell_eval` and
`filter_pred_by_true_combinations` helpers are inlined in the full script (no `pertx`
import). See `evaluate_50gene_hf.py` for their implementations.

---

## Step 6 — Generate Embedding UMAPs

The model produces 32-dimensional `X_scGPT_next` embeddings for every cell. We use
UMAP to project these to 2D and color by the predicted perturbation (`genotype_next`).

Two UMAP modes are available:

### Full s/t/p UMAP (when control label ∈ genotype_to_index)

When the control label is known to the model, we can run separate inference on source
(control), target (real perturbation), and predicted cells, then produce three plot types:
- **Per-perturbation UMAPs** — one plot per gene showing WT→WT, Gene→Gene, and WT→Gene
- **Global combined UMAP** — all perturbations on one plot
- **Split UMAP** — side-by-side in-silico predictions vs ground truth

In [ ]:
def get_s_t_p(adata_test, ctrl_label="WT"):
    """Split test data into source, target, and perturbation views."""
    s = adata_test[adata_test.obs.genotype == ctrl_label].copy()
    t = adata_test[adata_test.obs.genotype != ctrl_label].copy()
    p = adata_test[adata_test.obs.genotype == ctrl_label].copy()

    s.obs["genotype_next"] = [ctrl_label] * s.shape[0]
    t.obs["genotype_next"] = t.obs.genotype

    t_unique = t.obs.genotype.unique()
    n_pert = min(p.shape[0], 100 * len(t_unique))
    p.obs["genotype_next"] = np.random.permutation(np.concatenate([
        np.random.choice(t_unique, n_pert),
        np.random.choice(t_unique, max(0, p.shape[0] - n_pert)),
    ]))
    return s, t, p

# Run inference on each split
s, t, p = get_s_t_p(test_adata)
s_res = eval_testdata(
    model,
    adata_t=s,
    gene_ids=data_produced["gene_ids"],
    train_data_dict=data_produced,
    config=config,
    include_types=["cls"],
    logger=create_logger(),
    predict_expr=True,
    mvc_full_expr=True,      # produce full-gene expression
    sizefactor=True,          # use NB size factors
    sample=True,
)  # source embeddings
t_res = eval_testdata(
    model,
    adata_t=t,
    gene_ids=data_produced["gene_ids"],
    train_data_dict=data_produced,
    config=config,
    include_types=["cls"],
    logger=create_logger(),
    predict_expr=True,
    mvc_full_expr=True,      # produce full-gene expression
    sizefactor=True,          # use NB size factors
    sample=True,
)  # target embeddings
p_res = eval_testdata(
    model,
    adata_t=p,
    gene_ids=data_produced["gene_ids"],
    train_data_dict=data_produced,
    config=config,
    include_types=["cls"],
    logger=create_logger(),
    predict_expr=True,
    mvc_full_expr=True,      # produce full-gene expression
    sizefactor=True,          # use NB size factors
    sample=True,
) # predicted embeddings

# Generate UMAPs
plot_insilico_perturbations2(
    s_res, t_res, p_res,
    save_dir="results/EMB/50gene_hf/perttf_tiny_50gene",
    name="PERTTF 50gene",
)

The full `plot_insilico_perturbations2` function (in the companion script) produces
per-gene, global combined, and split-prediction-vs-ground-truth UMAP views.

---

## Step 7 — Generate Gene-Expression Heatmaps

After `cell_eval` runs, the `ESC_real_de.csv` file contains per-perturbation differential
expression statistics (fold change, FDR). We select the top genes by FDR and fold change,
compute expression deltas (perturbed − control mean) for both true and predicted data,
and plot side-by-side boxplots.

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.sparse import issparse
from pathlib import Path # Added missing import to prevent NameError

plt.close('all')
sns.reset_orig()
sns.set_style("ticks") # "ticks" style never has a background grid

# 1. Load the data
# Replace these filenames with your actual file paths
de_dir = Path('results/CELL_EVAL/50gene_hf')
csv_path =  de_dir / 'ESC_real_de.csv' 


deg_df = pd.read_csv(csv_path)
adata_pred = eval_results['pred']
adata_true = eval_results['real']
deg_df['fold_change'] = deg_df['target_mean'] - deg_df['reference_mean']
# Identify unique perturbations in the AnnData (excluding the control/WT)
# Adjust 'WT' to match your actual control label in the 'genotype' column
reference_label = 'WT'
perturbations = [p for p in adata_true.obs['genotype'].unique() if p != reference_label]


# Data structures to store mean fold changes
pred_heatmap_data = []
true_heatmap_data = []

# To keep the heatmap rows consistent, we'll track all unique top genes
all_top_genes = set()

for p in perturbations:
    # Get top 20 genes by |FC| (FDR < 0.05)
    pert_deg = deg_df[deg_df['target'] == p].copy()
    sig_genes = pert_deg[(pert_deg['fdr'] < 0.01) & 
        (~pert_deg['feature'].str.startswith('MT-'))].copy()
    
    #if sig_genes.empty:
     #   continue
        
    sig_genes['abs_fc'] = sig_genes['fold_change'].abs()
    top_genes = sig_genes.sort_values('abs_fc', ascending=False).head(20)['feature'].tolist()
    #top_genes = deg_df[deg_df['target'] == p].sort_values('fdr').head(20)['feature'].tolist()
    all_top_genes.update(top_genes)

    # Function to get mean delta for these genes
    def get_mean_deltas(adata, genes, perturbation):
        cells_p = adata[adata.obs['genotype'] == perturbation, genes]
        cells_ref = adata[adata.obs['genotype'] == reference_label, genes]
        
        # Convert to dense if necessary
        expr_p = cells_p.X.toarray() if issparse(cells_p.X) else cells_p.X
        expr_ref = cells_ref.X.toarray() if issparse(cells_ref.X) else cells_ref.X
        
        # Calculate Delta: Mean(Perturbed) - Mean(WT)
        # (This represents the fold change in log-space or absolute change)
        delta = expr_p.mean(axis=0) - expr_ref.mean(axis=0)
        return pd.Series(delta, index=genes)

    pred_heatmap_data.append(get_mean_deltas(adata_pred, top_genes, p).rename(p))
    true_heatmap_data.append(get_mean_deltas(adata_true, top_genes, p).rename(p))

# Convert to DataFrames (Rows = Genes, Columns = Perturbations)
df_pred_map = pd.concat(pred_heatmap_data, axis=1).fillna(0)
df_true_map = pd.concat(true_heatmap_data, axis=1).fillna(0)

# Ensure both maps have the same genes and order
all_genes_sorted = sorted(list(all_top_genes))
df_pred_map = df_pred_map.reindex(all_genes_sorted).fillna(0)
df_true_map = df_true_map.reindex(all_genes_sorted).fillna(0)

# 4. Dynamic Sizing
n_genes = len(all_genes_sorted)
n_perts = len(perturbations)

# Matplotlib figsize is (width, height)
# Columns = Perturbations (Width), Rows = Genes (Height)
fig_width = max(15, n_perts )   # Width scales with perturbations
fig_height = max(8, n_genes * 0.25)  # Height scales with genes

print(f"Width: {fig_width}, Height: {fig_height}")

# CRITICAL FIX: Use layout='constrained' instead of plt.tight_layout()
fig, (ax1, ax2) = plt.subplots(
    1, 2, 
    figsize=(fig_width, fig_height), 
    gridspec_kw={'width_ratios': [1, 1], 'wspace': 0.05},
    layout='constrained' 
)

# 5. Plotting
v_min = min(df_pred_map.values.min(), df_true_map.values.min())
v_max = max(df_pred_map.values.max(), df_true_map.values.max())

# Heatmap 1: True
sns.heatmap(
    df_true_map, 
    ax=ax1, 
    cmap='RdBu_r', 
    center=0, 
    vmin=-1.5, 
    vmax=1.5, 
    cbar=False,         
    linewidths=0,       
    rasterized=True     
)
ax1.tick_params(axis='x', labelsize=22, rotation=90)
ax1.set_title('True Expression Delta', fontsize = 34)
ax1.set_ylabel('DEGs', fontsize = 32)
ax1.grid(False)

# Heatmap 2: Predicted
sns.heatmap(
    df_pred_map, 
    ax=ax2, 
    cmap='RdBu_r', 
    center=0, 
    vmin=-1.5, 
    vmax=1.5, 
    cbar=False,         
    linewidths=0,       
    rasterized=True     
)
ax2.tick_params(axis='x', labelsize=22, rotation=90)
ax2.set_title('Predicted Expression Delta', fontsize = 34)
ax2.set_ylabel('')
ax2.set_yticks([])
ax2.grid(False)         
# --- ADD UNIFIED HORIZONTAL COLORBAR ---
mappable = ax1.collections[0]
cbar = fig.colorbar(
    mappable, 
    ax=[ax1, ax2],                
    orientation='horizontal',     
    shrink=0.6,                   # Slightly smaller to look cleaner horizontally
    aspect=20,                    # Makes the colorbar thinner so it doesn't take up too much vertical space
    pad=0.02,                     # FIX 2: Reduce the pad significantly 
    fraction=0.05                   
)
cbar.ax.set_title('Perturbation', fontsize=32, pad=15)
cbar.set_label('Mean Delta Expression', fontsize = 26)
# ---------------------------------------

# Removed plt.tight_layout() because layout='constrained' handles it better automatically
plt.savefig(de_dir /'comparison_heatmaps.png', dpi=300)
plt.show()

---

## Expected Outputs

After a successful run, the following files are produced:

```
results/
├── CELL_EVAL/
│   └── 50gene_hf/
│       │── ESC_real_de.csv        # per-perturbation DEG statistics
│       │── ESC_summary.csv        # summary metrics
│       │── ...
│       ├── comparison_heatmaps.png            # true expression AnnData
│
└── EMB/
    └── 50gene_hf/                    # cell metadata
        └── perttf_tiny_50gene/
            ├── umap_global.png           # global UMAP
            ├── umap_global_combined.png  # combined UMAP (s/t/p mode)
            ├── umap_global_split.png     # split UMAP (s/t/p mode)
            └── umap_<gene>.png           # per-gene UMAPs (s/t/p mode)
```

Expected shapes:
- Test data: ~57,000 cells × 4,485 genes
- `mvc_next_expr`: cells × 4,485 genes (after genotype filtering)
- `X_scGPT_next`: N cells × 32 dimensions
- Per-perturbation UMAPs: one PNG per gene in the top-20 set

---

## Troubleshooting

### `ModuleNotFoundError: No module named 'perttf'`

The `pertTF` package must be on `sys.path`. In the repo root, add:

In [ ]:
sys.path.insert(0, "pertTF")

Or use the absolute path to the `pertTF/` directory.

### `RuntimeError: stack expects a non-empty TensorList`

This occurs when `eval_testdata` receives an AnnData with 0 cells. Common causes:
- The control label (`WT`, `non-target`, etc.) is not present in the dataset
- All cells were filtered out by `genotype_next` not being in `genotype_to_index`

**Fix**: Auto-detect the control label (see Step 2) and use the fallback UMAP path
when the control label is not in `genotype_to_index` (see Step 6).

### `ESC_real_de.csv` not found

`cell_eval` writes this file inside subdirectories named by cell-type prefix
(e.g., `celltype_0/ESC_real_de.csv`). Use `Path.rglob("ESC_real_de.csv")` to search
recursively instead of assuming a top-level path.

### `flash_attn` or CUDA errors on H200

Set `amp=True` in the config. H200 GPUs require mixed-precision for FlashAttention:

In [ ]:
config.amp = True

### Model predicts the wrong genes

The model only predicts expression for genes in its training vocabulary (5,057 tokens,
including 3 special tokens → ~5,054 genes). Intersect the dataset's genes with
`model.vocab.stoi` before inference (see Step 2).

---
